# Training
Model dilatih pada GPU Kaggle. Dua eksperimen dibandingkan:

| Eksperimen | Model | Ukuran input | Tujuan |
|---|---|---|---|
| baseline | ResNet-18 | 224 | pembanding sederhana |
| final | ConvNeXt-Tiny | 384 | model utama |

Keduanya menggunakan bobot pre-trained ImageNet yang di-fine-tune pada seluruh lapisan. Pengaturan lengkap ada di `configs/baseline.yaml` dan `configs/config.yaml`.

## 0. Setup
Jika dijalankan di Kaggle, repository di-clone otomatis ke `/kaggle/working`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Violita-archive/document-image-classification.git"
REPO_NAME = "document-image-classification"

if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working") / REPO_NAME
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    try:
        import timm
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Root:", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.dataset import clean_train_data, load_metadata, make_loader, split_train_val
from src.model import load_checkpoint
from src.predict import make_submission, validate_submission
from src.train import fit
from src.transforms import denormalize, get_train_transforms
from src.utils import CLASS_NAMES, ensure_dirs, get_device, load_config, plot_history, resolve_data_dir, set_seed

FIG_DIR, METRIC_DIR, SUB_DIR = Path("results/figures"), Path("results/metrics"), Path("results/submissions")
ensure_dirs(FIG_DIR, METRIC_DIR, SUB_DIR, "checkpoint")

base_cfg = load_config("configs/baseline.yaml")
final_cfg = load_config("configs/config.yaml")
set_seed(final_cfg["seed"])

device = get_device()
print("PyTorch", torch.__version__, "| device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 1. Data
Pembersihan mengikuti hasil EDA: file yang tidak dapat dibuka dan grup gambar identik berlabel berbeda dibuang, sedangkan duplikat berlabel sama disisakan satu. Data bersih kemudian dibagi 90% train dan 10% validation secara stratified.

In [ ]:
DATA_DIR = resolve_data_dir(final_cfg["data_dir"])
print("Folder data:", DATA_DIR)

train_full, test_df = load_metadata(DATA_DIR)
clean_df, clean_summary = clean_train_data(train_full, n_workers=8)
display(pd.Series(clean_summary, name="jumlah").to_frame().T)

tr_df, va_df = split_train_val(clean_df, final_cfg["val_size"], final_cfg["seed"])
print(f"train: {len(tr_df):,} | validation: {len(va_df):,} | test: {len(test_df):,}")
display(pd.DataFrame({"train": tr_df["class_name"].value_counts(), "validation": va_df["class_name"].value_counts()}).reindex(CLASS_NAMES).T)

## 2. Contoh hasil augmentasi

In [ ]:
aug_loader = make_loader(tr_df.sample(8, random_state=1), get_train_transforms(final_cfg["img_size"]), 8, False, 0)
x, y = next(iter(aug_loader))
fig, axes = plt.subplots(1, 8, figsize=(20, 3.6))
for ax, img, lab in zip(axes, x, y):
    ax.imshow(denormalize(img))
    ax.set_title(CLASS_NAMES[lab], fontsize=8)
    ax.axis("off")
plt.suptitle(f"Contoh gambar setelah augmentasi ({final_cfg['img_size']}x{final_cfg['img_size']})")
plt.tight_layout()
plt.savefig(FIG_DIR / "train_augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Eksperimen 1: Baseline ResNet-18 (224)

In [ ]:
pd.Series(base_cfg, name="nilai").to_frame().T

In [ ]:
out_base = fit(base_cfg, tr_df, va_df, exp_name="baseline_resnet18")
print(f"Akurasi validation terbaik: {out_base['best_acc']:.4f} (epoch {out_base['best_epoch']}), waktu {out_base['minutes']:.1f} menit")
plot_history(out_base["history"], "Baseline ResNet-18 (224)", FIG_DIR / "curve_baseline_resnet18.png")

## 4. Eksperimen 2: ConvNeXt-Tiny (384)

In [ ]:
pd.Series(final_cfg, name="nilai").to_frame().T

In [ ]:
out_final = fit(final_cfg, tr_df, va_df, exp_name="final_convnext_tiny")
print(f"Akurasi validation terbaik: {out_final['best_acc']:.4f} (epoch {out_final['best_epoch']}), waktu {out_final['minutes']:.1f} menit")
plot_history(out_final["history"], "ConvNeXt-Tiny (384)", FIG_DIR / "curve_final_convnext_tiny.png")

## 5. Perbandingan eksperimen

In [ ]:
runs = {"baseline_resnet18": (base_cfg, out_base), "final_convnext_tiny": (final_cfg, out_final)}
rows = []
for name, (cfg, out) in runs.items():
    h = out["history"]
    rows.append({
        "eksperimen": name,
        "model": cfg["model_name"],
        "img_size": cfg["img_size"],
        "epoch_dijalankan": len(h),
        "epoch_terbaik": out["best_epoch"],
        "train_acc": h.loc[h["epoch"] == out["best_epoch"], "train_acc"].iat[0],
        "val_acc": out["best_acc"],
        "val_loss": h.loc[h["epoch"] == out["best_epoch"], "val_loss"].iat[0],
        "menit": out["minutes"],
    })
experiments = pd.DataFrame(rows)
experiments.to_csv(METRIC_DIR / "experiments.csv", index=False)
display(experiments)

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, (_, out) in runs.items():
    ax.plot(out["history"]["epoch"], out["history"]["val_acc"], marker="o", label=name)
ax.set(title="Akurasi validation per epoch", xlabel="epoch", ylabel="akurasi")
ax.grid(alpha=0.3)
ax.legend()
plt.savefig(FIG_DIR / "experiments_val_acc.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Prediksi validation
Probabilitas dari model terbaik tiap eksperimen disimpan untuk analisis di `03_evaluation.ipynb`.

In [ ]:
for name, (_, out) in runs.items():
    probs = out["val_probs"]
    pred_df = va_df[["id", "filename", "label"]].copy()
    pred_df["pred"] = probs.argmax(1)
    pred_df["confidence"] = probs.max(1)
    pred_df[[f"prob_{i}" for i in range(len(CLASS_NAMES))]] = probs
    pred_df.to_csv(METRIC_DIR / f"{name}_val_predictions.csv", index=False)
    print(f"{name}: akurasi validation {(pred_df['pred'] == pred_df['label']).mean():.4f}")

## 7. Submission

In [ ]:
for name, (_, out) in runs.items():
    model, cfg = load_checkpoint(out["ckpt_path"], device)
    sub = make_submission(model, cfg, test_df, SUB_DIR / f"submission_{name}.csv")
    validate_submission(sub, test_df)
    print(f"{name}: {len(sub)} baris disimpan")
    del model
    torch.cuda.empty_cache()

display(sub.head())
display(sub["label"].map(lambda i: CLASS_NAMES[i]).value_counts().to_frame("jumlah_prediksi").T)

## 8. Arsip hasil
Seluruh isi `results/` dan `checkpoint/` dikompres agar mudah diunduh dari tab Output Kaggle.

In [ ]:
import zipfile

archive = ROOT.parent / "hasil_training.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("results", "checkpoint"):
        for f in Path(folder).rglob("*"):
            if f.is_file():
                z.write(f)
print(archive, f"{archive.stat().st_size / 1e6:.1f} MB")